# Load data

In [1]:
def run_opt(budget,directory='',x_b_fix=False):

    ''' Script to run optimization
    After running set_up_optimization, parameters will be ready for input to AMPL script.
    This code loads the AMPL script, populates the parameters, and executes the optimization

    budget = budgetary limit
    
    directory = directory where to load data from (e.g., 'backend/Data/')
    
    x_b_fix = fixed building stock matrix, set to None if it is not fixed. If fixed, the analysis will run the event over a predefined 
    building stock'''

    # ======================================================
    # ============ Get previous building stock, if applicable
    # ======================================================
    
    run_fix = False
    if x_b_fix != False:
        try:
            x_b_fix = ampl.get_variable('x_b').to_pandas().unstack().values.reshape(x_b_0.shape)
            run_fix = True
        except: 
            print('No previous analyses run. Building stock will not be fixed...')

    
    # Load AMPL file
    
    ampl.reset()
    
    try:
        ampl.read("ARC_model.mod")
        
    except:
         ampl.read("..//ARC_model.mod")

    # Load neighborhoods
    neighborhoods = gp.read_file(rf'{directory}Neighborhoods/neighborhoods.shp')

    # Set up parameters for AMPL
    
    ampl.param['n_neighborhoods'] = neighborhoods.shape[0] # number of neighborhoods
    ampl.param['n_archetypes'] = x_b_0.shape[1] # number of building archetypes
    ampl.param['n_DS'] = 5 # number of damage states (including 0) 
    ampl.param['n_retrofits'] = 6 # number of possible retrofits (including no action and combinations)
    # (here, there is no action, R1/R2, shelter, and shelter + R1/R2)
    
    ampl.set['Q'] = permissible # permissible set of upgrades

    ampl.param['B'] = budget # budget
    
    ampl.param['n_injuries'] = casualty_costs.shape[0] # number of injury severity levels
    
    ampl.param['c_cas'] = casualty_costs # casualty costs 
    
    ampl.param['C_dr'] = C_dr # debris removal cost

    ampl.param['f_footprint'] = list(f_footprint) # building footprint factors

    # Initial building stock
    ampl.param['x_b_0'] = {
        (b,c,d): x_b_0[i][j][k]
        for i, b in enumerate(list(np.arange(x_b_0.shape[0])+1))
        for j, c in enumerate(list(np.arange(x_b_0.shape[1])+1))
        for k, d in enumerate(list(np.arange(x_b_0.shape[2])+1))
    }

    # Number of residents in each building type/neighborhood
    ampl.param['H'] = {
        (b,c): building_size[i][j]
        for i, b in enumerate(list(np.arange(building_size.shape[0])+1))
        for j, c in enumerate(list(np.arange(building_size.shape[1])+1))
    }

    # Cost of retrofits
    ampl.param['c_b_ret'] = {
        (b,c,d): c_b_ret[i][j][k]
        for i, b in enumerate(list(np.arange(c_b_ret.shape[0])+1))
        for j, c in enumerate(list(np.arange(c_b_ret.shape[1])+1))
        for k, d in enumerate(list(np.arange(c_b_ret.shape[2])+1))
    }
    
    # Building repair cost
    ampl.param['c_b_rep'] = {
        (b,c): c_b_rep[i][j]
        for i, b in enumerate(list(np.arange(c_b_rep.shape[0])+1))
        for j, c in enumerate(list(np.arange(c_b_rep.shape[1])+1))
    }

    # Damage state probability matrix
    ampl.param['p_DS'] = {
        (b,c,d,e): DS_matrix[i][j][k][l]
        for i, b in enumerate(list(np.arange(DS_matrix.shape[0])+1))
        for j, c in enumerate(list(np.arange(DS_matrix.shape[1])+1))
        for k, d in enumerate(list(np.arange(DS_matrix.shape[2])+1))
        for l, e in enumerate(list(np.arange(DS_matrix.shape[3])+1))
    }

    # Damage state probabiltiy matrix (for sheltered)    
    ampl.param['p_DS_for_casualties'] = {
        (b,c,d,e): DS_matrix_for_casualties[i][j][k][l]
        for i, b in enumerate(list(np.arange(DS_matrix_for_casualties.shape[0])+1))
        for j, c in enumerate(list(np.arange(DS_matrix_for_casualties.shape[1])+1))
        for k, d in enumerate(list(np.arange(DS_matrix_for_casualties.shape[2])+1))
        for l, e in enumerate(list(np.arange(DS_matrix_for_casualties.shape[3])+1))
    }

    ampl.param['f_pw'] = f_pw # present worth factor

    # Dislocation costs
    ampl.param['c_d'] = {
        (b,c,d): c_d [i][j][k]
        for i, b in enumerate(list(np.arange(c_d.shape[0])+1))
        for j, c in enumerate(list(np.arange(c_d.shape[1])+1))
        for k, d in enumerate(list(np.arange(c_d.shape[2])+1))
    }
    
    # Injury probability 
    ampl.param['p_i'] = {
        (b,c): casualty_rates [i][j]
        for i, b in enumerate(list(np.arange(casualty_rates.shape[0])+1))
        for j, c in enumerate(list(np.arange(casualty_rates.shape[1])+1))
    } 
    
    # Volume of building debris
    ampl.param['V_db'] = {
        (b,c,d): V_db [i][j][k]
        for i, b in enumerate(list(np.arange(V_db.shape[0])+1))
        for j, c in enumerate(list(np.arange(V_db.shape[1])+1))
        for k, d in enumerate(list(np.arange(V_db.shape[2])+1))
    }   

    # ============ Fix building stock
    
    if run_fix == True:
        
        with open(directory+"fix_commands.dat", "w") as file: # Create new .dat file of constraints
    
            
            for i in range(x_b_fix.shape[0]):
                for j in range(x_b_fix.shape[1]):
                    for k in range(x_b_fix.shape[2]): 
                        
                        file.write(f'fix x_b[{i+1},{j+1},{k+1}] := {x_b_fix[i][j][k]};\n') # Set fixed value
                        
        ampl.read(rf"{directory}fix_commands.dat") # load fixed constraints

        ampl.eval('drop budget_constraint;') # Drop some unneeded constraints

  

    ampl.option["solver"] = "C:\\xpressmp\\bin\\amplxpress.exe"
    # ampl.option["solver"] = 'XPRESS' # define solver
    ampl.setOption("solver_msg", 0)

    ampl.solve() # execute optimization